In [1]:
import numpy as np
import time
from dataclasses import dataclass
from typing import Callable, Optional
import matplotlib.pyplot as plt

In [2]:
from simulator import generate_hierarchical
from posteriors import *
from samplers_hierarchical import *

In [3]:
rng = np.random.default_rng(221)

print("Generating data")
ds = generate_hierarchical(rng=rng, truth_model = "power_law")
K = len(ds.seasons)
print(f"K = {K}, "
     f"True means = {ds.phi_k}")


Generating data
K = 10, True means = [37.0740292  27.82323994 24.08274659 26.33040153 40.39555597 30.25958186
 32.61164236 39.91307355 19.05233462 33.71709834]


In [4]:
param_names = []

for i in range(K):
    param_names.append(f"log_phi_{i+1}")

param_names.extend(["gamma", "log_eta", "delta", "mu_phi", "log_sigma_phi"])
param_names

['log_phi_1',
 'log_phi_2',
 'log_phi_3',
 'log_phi_4',
 'log_phi_5',
 'log_phi_6',
 'log_phi_7',
 'log_phi_8',
 'log_phi_9',
 'log_phi_10',
 'gamma',
 'log_eta',
 'delta',
 'mu_phi',
 'log_sigma_phi']

In [5]:

true_values = {
    **{f"log_phi_{k+1}": np.log(ds.phi_k[k]) for k in range(K)},
    "gamma": ds.true_params["gamma"],
    "log_eta": np.log(ds.true_params["eta"]),
    "delta": ds.true_params["delta"],
    "mu_phi": ds.true_params["mu_phi"],
    "log_sigma_phi": np.log(ds.true_params["sigma_phi"]),
}


In [6]:
mu_phi_true = ds.true_params["mu_phi"]
sigma_phi_true = ds.true_params["sigma_phi"]

z_k_true = (np.log(ds.phi_k) - mu_phi_true) / sigma_phi_true

theta_true = np.concatenate([
    z_k_true,
    [
        ds.true_params["gamma"],
        np.log(ds.true_params["eta"]),
        ds.true_params["delta"],
        mu_phi_true,
        np.log(sigma_phi_true),
    ],
])

theta_init = theta_true

In [7]:
prior_type = "lognormal_gamma"
parameterization = "noncentered"

def log_post(theta):
    return log_posterior_hierarchical(theta, ds.seasons, parameterization, prior_type)

def grad_log_post(theta):
    return grad_log_posterior_hierarchical(theta, ds.seasons, parameterization, prior_type)

In [8]:
rwmh_results = run_multiple_chains(
        run_rwmh,
        theta_init=theta_init,
        n_chains=4,
        init_strategy="jitter",
        init_scale=0.001,
        rng=rng,
        log_posterior_fn=log_post,
        n_iterations=100000,
        n_burnin=20000,
        adapt_until=20000,
        adapt_proposal=True,
        param_names=param_names,
    )

Iteration 5000/100000: accept rate = 0.264, scale = 1.468, elapsed = 0.9s
Iteration 10000/100000: accept rate = 0.250, scale = 0.997, elapsed = 1.9s
Iteration 15000/100000: accept rate = 0.242, scale = 0.565, elapsed = 2.9s
Iteration 20000/100000: accept rate = 0.237, scale = 0.460, elapsed = 3.9s
Iteration 25000/100000: accept rate = 0.241, scale = 0.460, elapsed = 4.9s
Iteration 30000/100000: accept rate = 0.246, scale = 0.460, elapsed = 6.0s
Iteration 35000/100000: accept rate = 0.246, scale = 0.460, elapsed = 6.9s
Iteration 40000/100000: accept rate = 0.248, scale = 0.460, elapsed = 7.9s
Iteration 45000/100000: accept rate = 0.251, scale = 0.460, elapsed = 8.8s
Iteration 50000/100000: accept rate = 0.252, scale = 0.460, elapsed = 9.8s
Iteration 55000/100000: accept rate = 0.252, scale = 0.460, elapsed = 10.8s
Iteration 60000/100000: accept rate = 0.251, scale = 0.460, elapsed = 11.8s
Iteration 65000/100000: accept rate = 0.252, scale = 0.460, elapsed = 12.7s
Iteration 70000/100000:

In [9]:
rwmh_cov = estimate_dense_precond_from_rwmh(rwmh_results, ridge=1e-6)

In [10]:
mala_results = run_multiple_chains(
        run_mala,
        theta_init=theta_init,
        n_chains=4,
        init_strategy="same",
        init_scale=0.001,
        rng=rng,
        log_posterior_fn=log_post,
        grad_log_posterior_fn=grad_log_post,
        n_iterations=100000,
        n_burnin=20000,
        step_size=1e-3,
        adapt_step=True,
        adapt_until=20000,
        target_accept=0.65,
        param_names=param_names,
        precond=rwmh_cov,
        adapt_precond=False,
        precond_type="dense",
        normalize_precond=True,
    )

Iteration 5000/100000: accept rate = 0.775, step_size = 0.0141991, elapsed = 1.9s
Iteration 10000/100000: accept rate = 0.717, step_size = 0.0118, elapsed = 3.8s
Iteration 15000/100000: accept rate = 0.687, step_size = 0.0090375, elapsed = 5.7s
Iteration 20000/100000: accept rate = 0.635, step_size = 0.000791233, elapsed = 7.7s
Iteration 25000/100000: accept rate = 0.702, step_size = 0.000791233, elapsed = 9.6s
Iteration 30000/100000: accept rate = 0.742, step_size = 0.000791233, elapsed = 11.5s
Iteration 35000/100000: accept rate = 0.777, step_size = 0.000791233, elapsed = 13.5s
Iteration 40000/100000: accept rate = 0.802, step_size = 0.000791233, elapsed = 15.4s
Iteration 45000/100000: accept rate = 0.815, step_size = 0.000791233, elapsed = 17.4s
Iteration 50000/100000: accept rate = 0.815, step_size = 0.000791233, elapsed = 19.4s
Iteration 55000/100000: accept rate = 0.826, step_size = 0.000791233, elapsed = 21.4s
Iteration 60000/100000: accept rate = 0.837, step_size = 0.000791233,

In [11]:
# print_diagnostics_multi({"RWMH": rwmh_results,"MALA": mala_results}, true_values = true_values)

In [14]:
print_diagnostics_multi_hierarchical(
    {"RWMH": rwmh_results, "MALA": mala_results},
    parameterization="noncentered",
    K=10,
    latent_display="raw",
    true_values=true_values,
)

save_traceplots_multi_hierarchical(
    mala_results,
    "traceplots_mala_noncentered_logphi.png",
    parameterization="noncentered",
    K=10,
    latent_display="log_phi",
)


Sampler      Chains  Accept%  Time(s)ESS(z_1)ESS(z_2)ESS(z_3)ESS(z_4)ESS(z_5)ESS(z_6)ESS(z_7)ESS(z_8)ESS(z_9)ESS(z_10)ESS(gamma)ESS(log_eta)ESS(delta)ESS(mu_phi)ESS(log_sigma_phi)
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
RWMH              4    0.246     77.8       1935       1146       1507       1505       1480       1150       1227       1584       2030       1017        821        606        657        278        573
MALA              4    0.649    154.4        756        777        758        854        658        688        747        637        594        571        604        349        384        446        484

Sampler      Chains  Accept%  Time(s)Rhat(z_1)Rhat(z_2)Rhat(z_3)Rhat(z_4)Rhat(z_5)Rhat(z_6)Rhat(z_7)Rhat(z_8)Rhat(z_9)Rhat(z_10)Rhat(gamma)Rhat(log_eta)Rhat(delta)Rhat(mu_phi)Rhat(log_sigma_phi)


In [13]:
log_posterior_hierarchical(theta_true, ds.seasons, "centered")

np.float64(8421347.57997435)

In [16]:
theta_mala = np.array([-0.1009,0.3353,0.0046-0.0057,0.3186,-0.1673,-0.1265,
                      -0.1843,-0.1314,-0.1314,-0.1431,2.4964,6.2411,3.6875,1.5748,-1.4372])
log_posterior_hierarchical(theta_mala, ds.seasons, "centered")

np.float64(8421803.987571375)